# Exploring the common voting space

Thin viewer over the pipeline outputs. Run `python -m src.pipeline --chamber both` from the project root first, then execute cells top to bottom. (Run jupyter from the project root so `src` imports.)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import viz

viz.apply_style()
CHAMBER = "House"  # or "Senate"
pos = pd.read_csv(ROOT / f"output/positions_{CHAMBER}.csv")
diag = pd.read_csv(ROOT / f"output/diagnostics_{CHAMBER}.csv", index_col=0)
print(f"{len(pos):,} member-congress positions, {pos['icpsr'].nunique():,} members, "
      f"congresses {pos['congress'].min()}\u2013{pos['congress'].max()}")
pos.head()

In [ ]:
def show_congress(t: int):
    """One congress in the common space, colored by bloc lineage."""
    sub = pos[pos["congress"] == t]
    colors = viz._bloc_colors(pos)
    fig, ax = plt.subplots(figsize=(6, 5))
    for b, g in sub.groupby("bloc"):
        ax.scatter(g["dim1"], g["dim2"], s=18, c=colors[int(b)], alpha=0.85,
                   linewidths=0.4, edgecolors=viz.SURFACE, label=f"bloc {b}")
    ax.set_title(f"{CHAMBER} \u00b7 Congress {t} \u00b7 {viz.first_year(t)}\u2013{viz.first_year(t) + 2}")
    ax.set_xlabel("dimension 1")
    ax.set_ylabel("dimension 2")
    ax.legend()
    plt.show()

show_congress(119)
show_congress(88)  # civil-rights era: watch the second dimension open up

In [ ]:
# Polarization over time: separation of the two k=2 clusters
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(diag.index, diag["separation_k2"], color=viz.LINE_SLOTS[0], lw=2)
ax.set_title(f"{CHAMBER}: two-cluster separation over time")
ax.set_xlabel("congress")
plt.show()

diag[["n_members", "n_rollcalls", "k", "silhouette_k2", "separation_k2",
      "n_anchors", "anchor_resid"]].describe().round(2)

In [ ]:
# Trajectory of any member: pick an icpsr from pos (long careers are most fun)
spans = pos.groupby("icpsr")["congress"].agg(["count", "min", "max"])
spans.sort_values("count", ascending=False).head(10)

In [ ]:
def show_member(icpsr: int):
    g = pos[pos["icpsr"] == icpsr].sort_values("congress")
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(pos["dim1"], pos["dim2"], s=4, c=viz.MUTED, alpha=0.06, linewidths=0)
    ax.plot(g["dim1"], g["dim2"], color=viz.LINE_SLOTS[0], lw=2, marker="o", ms=4,
            mec=viz.SURFACE, mew=0.5)
    ax.annotate("start", (g.iloc[0]["dim1"], g.iloc[0]["dim2"]), color=viz.INK_2,
                textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_title(f"icpsr {icpsr}: {viz.first_year(g['congress'].min())}\u2013"
                 f"{viz.first_year(g['congress'].max()) + 2}")
    plt.show()

show_member(int(spans["count"].idxmax()))

**Next steps** (deferred by design): join Voteview `HSall_members.csv` on `icpsr` to name members, color by real party, and orient the axes; add `HSall_rollcalls.csv` to interpret dimensions by issue area.